# 6D MENT reconstruction

In [ ]:
import math
import os
import sys

import numpy as np
import matplotlib.pyplot as plt
import torch
from ipywidgets import interact
from skimage.transform import downscale_local_mean

from bmadx.bmad_torch.track_torch import Beam
from bmadx.bmad_torch.track_torch import TorchLattice
from bmadx.constants import PI
from bmadx.distgen_utils import create_beam
from bmadx.plot import plot_projections

from phase_space_reconstruction.diagnostics import ImageDiagnostic
from phase_space_reconstruction.modeling import NNTransform
from phase_space_reconstruction.modeling import InitialBeam
from phase_space_reconstruction.modeling import PhaseSpaceReconstructionModel
from phase_space_reconstruction.modeling import PhaseSpaceReconstructionModel3D_2screens
from phase_space_reconstruction.train import train_3d_scan_2screens
from phase_space_reconstruction.utils import split_2screen_dset
from phase_space_reconstruction.virtual.beamlines import quad_tdc_bend

from analysis_scripts import plot_3d_scan_data_2screens
from analysis_scripts import plot_3d_scan_data_2screens_contour
from analysis_scripts import create_clipped_dset

from plotting import plot_corner

import ment
from ment_utils import TorchLatticeTransform
from ment_utils import LatticeFactory

## Setup

In [ ]:
ndim = 6

In [ ]:
output_dir = None

## Load data

In [ ]:
data = torch.load("clipped_dset.pt")

In [ ]:
train_data, test_data = split_2screen_dset(data)

In [ ]:
train_data.images.shape

In [ ]:
train_data.params.shape

## Create GPSR model

Create lattice and diagnostics.

In [ ]:
# Create diagnostic beamlines
p0c = 43.3e06
lattice0 = quad_tdc_bend(p0c=p0c, dipole_on=False)
lattice1 = quad_tdc_bend(p0c=p0c, dipole_on=True)

# Scan over quad strength, tdc on/off and dipole on/off
scan_ids = [0, 2, 4]

# Create diagnostic screens
def make_screen(size: float, pixels: int) -> ImageDiagnostic:
    edges = torch.linspace(-size / 2.0, size / 2.0, pixels)
    bandwidth = (edges[1] - edges[0]) / 2.0
    return ImageDiagnostic(edges, edges, bandwidth)

npix = 150
screen0_size = 30.22e-03 * npix / 700.0
screen1_size = 26.96e-03 * npix / 700.0
screen0 = make_screen(screen0_size, npix)
screen1 = make_screen(screen1_size, npix)

## Create MENT-GPSR interface

Create lattice factor (returns lattice with specific parameters).

In [ ]:
lattice_factory = LatticeFactory(
    lattice0,
    lattice1,
    scan_ids=[0, 2, 4],
    beam_kws=dict(p0c=torch.tensor(p0c)),
)

Flatten the parameters list.

In [ ]:
nmeas = np.prod(train_data.params.shape[:-1])
nmeas = int(nmeas)
params_list = train_data.params.reshape((nmeas, 3))

# Shorten for now
nmeas = 4
params_list = params_list[:nmeas]

Create list of NumPy array transformations.

In [ ]:
transforms = []
for index, params in enumerate(params_list):
    dipole_on = params[2] < 1.00e-13
    transform = lattice_factory.make_transform(params=params, dipole_on=dipole_on)
    transforms.append(transform)

Create list of NumPy array measurements.

In [ ]:
projections = train_data.images.clone()
projections = projections.numpy()
projections = projections.sum(axis=3)  # average over multi-shot images
projections = projections.reshape((math.prod(train_data.params.shape[:-1]), projections.shape[-2], projections.shape[-1]))
projections = projections[:nmeas]
projections = [projection for projection in projections]

for index in range(len(projections)):
    factor = 2
    if factor > 1:
        projections[index] = downscale_local_mean(projections[index], (factor, factor))

projections = [[projection] for projection in projections]

In [ ]:
@interact(i=(0, len(projections) - 1))
def plot_projection(i):
    fig, ax = plt.subplots(figsize=(3, 3))
    ax.pcolormesh(projections[i][0].T)
    plt.show()

Make list of diagnostic functions.

In [ ]:
def make_ment_diagnostic(screen: ImageDiagnostic, **kws) -> ment.diag.HistogramND:
    bin_coords = [
        screen.bins_x.detach().numpy(),
        screen.bins_y.detach().numpy(),
    ]
    bin_edges = [ment.grid.coords_to_edges(c) for c in bin_coords]
    diagnostic = ment.diag.HistogramND(edges=bin_edges, axis=(0, 2), **kws)
    return diagnostic

In [ ]:
diagnostics = []
for index in range(len(transforms) // 2):
    for screen in [screen0, screen1]:
        diagnostic = make_ment_diagnostic(screen, blur=1.0)
        diagnostics.append([diagnostic])

In [ ]:
print(len(transforms))
print(len(diagnostics), len(diagnostics[0]))
print(len(projections), len(projections[0]))

## Create MENT model

In [ ]:
prior_scale = np.array([0.005, 0.005, 0.005, 0.005, 0.0005, 0.010])
prior_scale *= 0.001
prior = ment.prior.GaussianPrior(
    ndim=ndim,
    scale=prior_scale,
)

In [ ]:
grid_xmax = [0.005, 0.005, 0.005, 0.005, 0.0005, 0.010]
grid_xmax = np.array(grid_xmax)
grid_limits = list(zip(-grid_xmax, grid_xmax))

grid_shape = ndim * [15]
grid_shape = tuple(grid_shape)

# sampler = ment.samp.GridSampler(
#     grid_limits=grid_limits,
#     grid_shape=grid_shape,
# )

In [ ]:
burnin = 500
nchains = 500

proposal_cov = 0.10 * np.eye(ndim)
proposal_cov *= grid_xmax**2

start_points = np.random.multivariate_normal(
    np.zeros(ndim),
    np.eye(ndim) * 1.0,
    size=nchains,
)
# start_points *= grid_xmax
start_points *= 0.001

sampler = ment.samp.MetropolisHastingsSampler(
    ndim=ndim,
    chains=nchains,
    proposal_cov=proposal_cov,
    start=start_points,
    burnin=burnin,
    shuffle=True,
    verbose=True,
)

Create MENT model.

In [ ]:
rec_model = ment.MENT(
    ndim=ndim,
    transforms=transforms,
    projections=projections,
    diagnostics=diagnostics,
    prior=prior,
    sampler=sampler,
    nsamp=200_000,
    mode="sample",
    verbose=True,
)

In [ ]:
X = rec_model.sample(100_000)

In [ ]:
X.max(axis=0)

In [ ]:
X.std(axis=0)

In [ ]:
plt.hist(X[:, 0], bins=50);

In [ ]:
def plot_proj(projections_pred: list[np.ndarray], projections_meas: list[np.ndarray]):
    ncols = min(10, len(projections_pred))
    nrows = 2
    
    fig, axs = plt.subplots(
        ncols=ncols,
        nrows=nrows,
        figsize=(ncols * 1.0, nrows * 1.0),
        gridspec_kw=dict(hspace=0, wspace=0),
    )
    for j in range(ncols):
        axs[0, j].pcolormesh(projections_pred[j].T)
        axs[1, j].pcolormesh(projections_meas[j].T)
    for ax in axs.ravel():
        ax.set_xticks([])
        ax.set_yticks([])
        for loc in ax.spines:
            ax.spines[loc].set_visible(False)
    return fig, axs

In [ ]:
def plot_dist(X_pred: np.ndarray) -> None:
    fig, axs = plt.subplots(ncols=6, nrows=6, figsize=(8, 8), sharex=False, sharey=False)
    plot_corner(X_pred * 1000.0, bins=64, axs=axs)
    return fig, axs

In [ ]:
def plot_rec_model(rec_model: ment.MENT, nsamp: int):
    figs = []
    
    X_pred = rec_model.sample(nsamp)
    
    fig, ax = plot_dist(X_pred)
    figs.append(fig)
    
    projections_pred = ment.sim.forward(X_pred, transforms, diagnostics)
    projections_pred = ment.utils.unravel(projections_pred)
    projections_meas = ment.utils.unravel(rec_model.projections)
    
    fig, ax = plot_proj(projections_pred, projections_meas)
    figs.append(fig)

    return figs

In [ ]:
for epoch in range(10):
    print("epoch =", epoch)
    
    if epoch > 0:
        rec_model.gauss_seidel_step(
            learning_rate=0.75
        )
        
    plot_rec_model(rec_model, nsamp=100_000)
    plt.show()